In [3]:
# 1. Zaroori tools (Libraries) import kar rahe hain
import pandas as pd      
import numpy as np       
import warnings
warnings.filterwarnings('ignore') 

print("Libraries imported successfully! Naya dataset load ho raha hai...")

# 2. Asli IMDB Dataset ka NAYA 100% WORKING LINK
url = "https://raw.githubusercontent.com/Ankit152/IMDB-sentiment-analysis/master/IMDB-Dataset.csv"

try:
    # Pandas ka use karke CSV file ko read kar rahe hain
    df = pd.read_csv(url)
    
    print("\nDataset loaded successfully! Total Reviews:", df.shape[0])
    
    # Data ke shuruati 5 rows print kar rahe hain
    display(df.head())

except Exception as e:
    print("Error loading dataset:", e)

Libraries imported successfully! Naya dataset load ho raha hai...

Dataset loaded successfully! Total Reviews: 50000


,review,sentiment
0,One of the other reviewers has mentioned that ...,positive
1,A wonderful little production. <br /><br />The...,positive
2,I thought this was a wonderful way to spend ti...,positive
3,Basically there's a family where a little boy ...,negative
4,"Petter Mattei's ""Love in the Time of Money"" is...",positive


In [4]:
# 1. Text cleaning ke zaroori tools
import re
import nltk
from nltk.corpus import stopwords

# Faltu words (stopwords) ki dictionary download kar rahe hain
nltk.download('stopwords')
stop_words = set(stopwords.words('english'))

# 2. Ek Safai machine (function) bana rahe hain
def clean_text(text):
    # a. HTML tags hatana (jaise <br />)
    text = re.sub(r'<.*?>', ' ', text)
    # b. Sirf alphabets rakhna, baaki numbers/symbols hatana
    text = re.sub(r'[^a-zA-Z\s]', '', text)
    # c. Sab words ko chote aksharon (lowercase) mein badalna
    text = text.lower()
    # d. Faltu words (the, is, in) hatana
    words = text.split()
    words = [w for w in words if w not in stop_words]
    return ' '.join(words)

print("Safai shuru ho gayi hai... 50,000 reviews hain toh 15-30 seconds lagenge, thoda wait karna!")

# 3. Apne data par is machine ko chala rahe hain
df['cleaned_review'] = df['review'].apply(clean_text)

print("\nData Cleaning Complete! Niche table dekho, naya saaf column ban gaya hai:")
display(df[['review', 'cleaned_review', 'sentiment']].head())

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\arora\AppData\Roaming\nltk_data...
[nltk_data]   Unzipping corpora\stopwords.zip.


Safai shuru ho gayi hai... 50,000 reviews hain toh 15-30 seconds lagenge, thoda wait karna!

Data Cleaning Complete! Niche table dekho, naya saaf column ban gaya hai:


,review,cleaned_review,sentiment
0,One of the other reviewers has mentioned that ...,one reviewers mentioned watching oz episode yo...,positive
1,A wonderful little production. <br /><br />The...,wonderful little production filming technique ...,positive
2,I thought this was a wonderful way to spend ti...,thought wonderful way spend time hot summer we...,positive
3,Basically there's a family where a little boy ...,basically theres family little boy jake thinks...,negative
4,"Petter Mattei's ""Love in the Time of Money"" is...",petter matteis love time money visually stunni...,positive


In [5]:
# 1. Machine Learning ke zaroori tools import kar rahe hain
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

print("Step 1: English text ko numbers mein badal rahe hain (TF-IDF)...")

# Sirf top 5000 sabse important words le rahe hain taaki process fast ho
vectorizer = TfidfVectorizer(max_features=5000)
X = vectorizer.fit_transform(df['cleaned_review'])

# Target (jo humein predict karna hai: positive ya negative)
y = df['sentiment']

print("Step 2: Data ko Train (80%) aur Test (20%) mein baant rahe hain...")
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"Training data size: {X_train.shape[0]} reviews | Testing data size: {X_test.shape[0]} reviews")

print("\nStep 3: AI Model ko Train kar rahe hain (Logistic Regression)... wait karna thoda...")
# Logistic Regression text classification ke liye bohot solid model hota hai
model = LogisticRegression()
model.fit(X_train, y_train)

print("\nStep 4: Model ka Test le rahe hain...")
y_pred = model.predict(X_test)
accuracy = accuracy_score(y_test, y_pred)

print("=========================================")
print(f"🎉 BINGO! Model Training Complete! 🎉")
print(f"🏆 Aapke AI Model ki Accuracy hai: {accuracy * 100:.2f}%")
print("=========================================")

Step 1: English text ko numbers mein badal rahe hain (TF-IDF)...
Step 2: Data ko Train (80%) aur Test (20%) mein baant rahe hain...
Training data size: 40000 reviews | Testing data size: 10000 reviews

Step 3: AI Model ko Train kar rahe hain (Logistic Regression)... wait karna thoda...

Step 4: Model ka Test le rahe hain...
🎉 BINGO! Model Training Complete! 🎉
🏆 Aapke AI Model ki Accuracy hai: 88.92%


In [6]:
# 1. Pehle apne model ka test lete hain apni khud ki lines par!
def check_my_review(review_text):
    # Pehle text ko saaf karo
    cleaned = clean_text(review_text)
    # Fir usko numbers mein badlo (vectorize)
    vectorized = vectorizer.transform([cleaned])
    # Model se pucho ki kya lagta hai
    prediction = model.predict(vectorized)[0]
    
    print(f"Aapka Review: '{review_text}'")
    if prediction == 'positive':
        print("🤖 AI Result: POSITIVE 😊👍\n")
    else:
        print("🤖 AI Result: NEGATIVE 😡👎\n")

print("--- AI TESTING CHALU HAI ---\n")
# Test 1
check_my_review("This movie was absolutely amazing, I loved the acting!")
# Test 2
check_my_review("Terrible waste of time, totally boring and bad story.")


# 2. Ab is trained AI ko file mein SAVE kar rahe hain Web App ke liye
import pickle

print("--- MODEL SAVE HO RAHA HAI ---")
# Model ka 'brain' save kar rahe hain
with open('sentiment_model.pkl', 'wb') as file:
    pickle.dump(model, file)

# Model ka 'translator' (vectorizer) save kar rahe hain
with open('vectorizer.pkl', 'wb') as file:
    pickle.dump(vectorizer, file)

print("✅ Badaai Ho! 'sentiment_model.pkl' aur 'vectorizer.pkl' save ho gaye hain.")
print("Ab hum in files ko use karke apni Streamlit Website banayenge!")

--- AI TESTING CHALU HAI ---

Aapka Review: 'This movie was absolutely amazing, I loved the acting!'
🤖 AI Result: POSITIVE 😊👍

Aapka Review: 'Terrible waste of time, totally boring and bad story.'
🤖 AI Result: NEGATIVE 😡👎

--- MODEL SAVE HO RAHA HAI ---
✅ Badaai Ho! 'sentiment_model.pkl' aur 'vectorizer.pkl' save ho gaye hain.
Ab hum in files ko use karke apni Streamlit Website banayenge!
